## Name: Asit Jain
## Roll No: M25DE1049
## Assignment 3 - MLBD

# Part 5: Learning-Based Recommender Systems
## Task 9: Reinforcement Learning in Recommender Systems

Implement RL-based recommenders that optimize long-term engagement via exploration-exploitation tradeoffs.

```
State (user history) -> Agent selects Action (movie) -> Observe Reward (rating)
                     -> Update Policy -> Repeat
```

**Approaches:** ε-Greedy Multi-Armed Bandit, UCB Bandit, Q-Learning Agent

**Dataset:** MovieLens ml-latest-small

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install',
    'pandas', 'numpy', 'scikit-learn', 'matplotlib', '-q'])

import os, zipfile, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict
from sklearn.metrics import mean_squared_error
print('All imports successful!')

### Step 1: Load the Dataset

In [ ]:
url = 'https://files.grouplens.org/datasets/movielens/ml-latest-small.zip'
zip_path = 'ml-latest-small.zip'
extract_dir = 'ml-latest-small'
if not os.path.exists(extract_dir):
    urllib.request.urlretrieve(url, zip_path)
    with zipfile.ZipFile(zip_path, 'r') as z: z.extractall('.')

movies = pd.read_csv(os.path.join(extract_dir, 'movies.csv'))
ratings = pd.read_csv(os.path.join(extract_dir, 'ratings.csv'))
print(f'Movies: {len(movies)}, Ratings: {len(ratings)}')
print(f'Users: {ratings["userId"].nunique()}, Rating range: {ratings["rating"].min()}-{ratings["rating"].max()}')

### Step 2: Define the Recommendation Environment

Build a simulation environment where:
- **State**: User's past interactions (rated movies)
- **Action**: Recommend a movie
- **Reward**: rating ≥ 4 → +1 (positive), rating < 4 → -1 (negative), no rating → 0 (neutral)

In [ ]:
# Build a lookup: (userId, movieId) -> reward
# rating >= 4 -> +1 (positive), rating < 4 -> -1 (negative)
reward_lookup = {}
for _, row in ratings.iterrows():
    r = 1.0 if row['rating'] >= 4.0 else -1.0
    reward_lookup[(int(row['userId']), int(row['movieId']))] = r

# Get all unique movie IDs (arms)
all_movie_ids = movies['movieId'].values
movie_id_to_idx = {mid: i for i, mid in enumerate(all_movie_ids)}
n_arms = len(all_movie_ids)

# Per-user rated sets for filtering
user_rated = ratings.groupby('userId')['movieId'].apply(set).to_dict()

# Select a subset of active users for simulation
user_counts = ratings.groupby('userId').size()
active_users = user_counts[user_counts >= 50].index.tolist()
print(f'Arms (movies): {n_arms}')
print(f'Active users (>=50 ratings): {len(active_users)}')

def get_reward(user_id, movie_id):
    """Simulate reward from the ratings dataset."""
    return reward_lookup.get((user_id, movie_id), 0.0)

### Step 3: ε-Greedy Multi-Armed Bandit

Each movie is an arm. The bandit maintains estimated rewards per arm and uses ε-greedy exploration:
- With probability ε, pick a random movie (explore)
- With probability 1-ε, pick the movie with highest estimated reward (exploit)

In [ ]:
class EpsilonGreedyBandit:
    """ε-Greedy Multi-Armed Bandit for movie recommendations."""
    def __init__(self, n_arms, epsilon=0.1):
        self.n_arms = n_arms
        self.epsilon = epsilon
        self.q_values = np.zeros(n_arms)      # estimated reward per arm
        self.arm_counts = np.zeros(n_arms)     # times each arm was pulled
        self.total_reward = 0.0
        self.rewards_history = []

    def select_arm(self, valid_mask=None):
        """Select an arm using ε-greedy strategy."""
        if valid_mask is None:
            valid_mask = np.ones(self.n_arms, dtype=bool)
        valid_idx = np.where(valid_mask)[0]
        if len(valid_idx) == 0:
            return np.random.randint(self.n_arms)
        if np.random.random() < self.epsilon:
            return np.random.choice(valid_idx)  # explore
        else:
            valid_q = self.q_values[valid_idx]
            return valid_idx[np.argmax(valid_q)]  # exploit

    def update(self, arm, reward):
        """Update estimated reward using incremental mean."""
        self.arm_counts[arm] += 1
        n = self.arm_counts[arm]
        self.q_values[arm] += (reward - self.q_values[arm]) / n
        self.total_reward += reward
        self.rewards_history.append(reward)

print('ε-Greedy Bandit defined.')

### Step 4: UCB (Upper Confidence Bound) Bandit

UCB prioritizes less-explored movies that may have high rewards. It balances exploration and exploitation using a confidence term.

In [ ]:
class UCBBandit:
    """Upper Confidence Bound Bandit for movie recommendations."""
    def __init__(self, n_arms, c=2.0):
        self.n_arms = n_arms
        self.c = c  # exploration parameter
        self.q_values = np.zeros(n_arms)
        self.arm_counts = np.zeros(n_arms)
        self.total_steps = 0
        self.total_reward = 0.0
        self.rewards_history = []

    def select_arm(self, valid_mask=None):
        """Select arm using UCB1 formula."""
        if valid_mask is None:
            valid_mask = np.ones(self.n_arms, dtype=bool)
        valid_idx = np.where(valid_mask)[0]
        if len(valid_idx) == 0:
            return np.random.randint(self.n_arms)
        # Pull each valid arm at least once
        unpulled = valid_idx[self.arm_counts[valid_idx] == 0]
        if len(unpulled) > 0:
            return np.random.choice(unpulled)
        # UCB1 score
        ucb_scores = self.q_values[valid_idx] + self.c * np.sqrt(
            np.log(self.total_steps + 1) / self.arm_counts[valid_idx])
        return valid_idx[np.argmax(ucb_scores)]

    def update(self, arm, reward):
        """Update estimated reward."""
        self.arm_counts[arm] += 1
        n = self.arm_counts[arm]
        self.q_values[arm] += (reward - self.q_values[arm]) / n
        self.total_steps += 1
        self.total_reward += reward
        self.rewards_history.append(reward)

print('UCB Bandit defined.')

### Step 5: Run MAB Simulation

Simulate recommendations for active users. Each step: agent picks a movie, observes reward from the dataset.

In [ ]:
def run_bandit_simulation(bandit_class, bandit_kwargs, users, n_steps=200):
    """Run bandit simulation across users."""
    all_rewards = []
    explore_counts = 0
    exploit_counts = 0
    cumulative = []

    for uid in users:
        bandit = bandit_class(n_arms=n_arms, **bandit_kwargs)
        rated = user_rated.get(uid, set())
        # Create valid mask: only movies this user has rated (so we can get reward)
        valid_mask = np.array([mid in rated for mid in all_movie_ids])
        if valid_mask.sum() == 0:
            continue

        for step in range(min(n_steps, int(valid_mask.sum()))):
            arm = bandit.select_arm(valid_mask)
            movie_id = all_movie_ids[arm]
            reward = get_reward(uid, movie_id)
            bandit.update(arm, reward)
            all_rewards.append(reward)
            # Track explore vs exploit (for ε-greedy)
            if hasattr(bandit, 'epsilon'):
                if np.random.random() < bandit.epsilon:
                    explore_counts += 1
                else:
                    exploit_counts += 1
            # Remove this movie so it's not recommended again
            valid_mask[arm] = False

    # Compute cumulative average reward
    cum_avg = np.cumsum(all_rewards) / np.arange(1, len(all_rewards) + 1)
    return all_rewards, cum_avg, explore_counts, exploit_counts

# Run ε-Greedy (ε=0.1)
np.random.seed(42)
eg_rewards, eg_cum, eg_explore, eg_exploit = run_bandit_simulation(
    EpsilonGreedyBandit, {'epsilon': 0.1}, active_users[:50], n_steps=100)

# Run UCB
np.random.seed(42)
ucb_rewards, ucb_cum, _, _ = run_bandit_simulation(
    UCBBandit, {'c': 2.0}, active_users[:50], n_steps=100)

print(f'ε-Greedy: {len(eg_rewards)} steps, avg reward: {np.mean(eg_rewards):.4f}')
print(f'UCB:      {len(ucb_rewards)} steps, avg reward: {np.mean(ucb_rewards):.4f}')
print(f'\nε-Greedy explore/exploit ratio: ~10% explore (ε=0.1)')

### Step 6: Visualize MAB Cumulative Rewards

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Cumulative average reward
axes[0].plot(eg_cum, label='ε-Greedy (ε=0.1)', alpha=0.8)
axes[0].plot(ucb_cum, label='UCB (c=2)', alpha=0.8)
axes[0].set_xlabel('Step')
axes[0].set_ylabel('Cumulative Average Reward')
axes[0].set_title('MAB: Cumulative Average Reward Over Time')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Reward distribution
axes[1].hist(eg_rewards, bins=3, alpha=0.6, label='ε-Greedy', edgecolor='black')
axes[1].hist(ucb_rewards, bins=3, alpha=0.6, label='UCB', edgecolor='black')
axes[1].set_xlabel('Reward')
axes[1].set_ylabel('Count')
axes[1].set_title('Reward Distribution')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Step 7: Q-Learning Agent

A Q-Learning agent that learns a Q-table mapping (state, action) pairs to expected rewards.

**Q-update rule:**
$$Q(s, a) = Q(s, a) + \alpha \left[ r + \gamma \max_{a'} Q(s', a') - Q(s, a) \right]$$

- State: discretized user genre preference bucket
- Action: movie index
- α = 0.1 (learning rate), γ = 0.9 (discount factor)

In [ ]:
# Build genre preferences per user as state representation
all_genres = sorted(set(g for gs in movies['genres'] for g in gs.split('|') if g != '(no genres listed)'))
genre_cols = [f'g_{g}' for g in all_genres]
for genre in all_genres:
    movies[f'g_{genre}'] = movies['genres'].apply(lambda x, g=genre: 1.0 if g in x.split('|') else 0.0)

# Movie genre lookup
movie_genre_map = movies.set_index('movieId')[genre_cols]

# Compute user's dominant genre as a discrete state
def get_user_state(user_id, rated_movies):
    """Discretize user state based on dominant genre preference."""
    if not rated_movies:
        return 0  # unknown state
    genre_scores = np.zeros(len(all_genres))
    for mid in rated_movies:
        if mid in movie_genre_map.index:
            r = reward_lookup.get((user_id, mid), 0)
            if r > 0:  # only count positively rated
                genre_scores += movie_genre_map.loc[mid].values
    if genre_scores.sum() == 0:
        return 0
    return int(np.argmax(genre_scores)) + 1  # 1-indexed genre bucket

n_states = len(all_genres) + 1  # 0 = unknown, 1..n = genre buckets
print(f'States: {n_states} (genre-based buckets), Actions: {n_arms} (movies)')

In [ ]:
class QLearningAgent:
    """Q-Learning agent for movie recommendations."""
    def __init__(self, n_states, n_actions, alpha=0.1, gamma=0.9, epsilon=0.1):
        self.n_states = n_states
        self.n_actions = n_actions
        self.alpha = alpha    # learning rate
        self.gamma = gamma    # discount factor
        self.epsilon = epsilon
        # Sparse Q-table using defaultdict (state -> action values)
        self.q_table = defaultdict(lambda: np.zeros(n_actions))
        self.rewards_history = []
        self.total_reward = 0.0

    def select_action(self, state, valid_mask=None):
        """ε-greedy action selection."""
        if valid_mask is None:
            valid_mask = np.ones(self.n_actions, dtype=bool)
        valid_idx = np.where(valid_mask)[0]
        if len(valid_idx) == 0:
            return np.random.randint(self.n_actions)
        if np.random.random() < self.epsilon:
            return np.random.choice(valid_idx)
        q_vals = self.q_table[state][valid_idx]
        return valid_idx[np.argmax(q_vals)]

    def update(self, state, action, reward, next_state, valid_mask=None):
        """Q-learning update rule."""
        if valid_mask is not None:
            valid_idx = np.where(valid_mask)[0]
            if len(valid_idx) > 0:
                max_next_q = np.max(self.q_table[next_state][valid_idx])
            else:
                max_next_q = 0.0
        else:
            max_next_q = np.max(self.q_table[next_state])
        # Q(s,a) = Q(s,a) + α[r + γ·max Q(s',a') - Q(s,a)]
        td_target = reward + self.gamma * max_next_q
        td_error = td_target - self.q_table[state][action]
        self.q_table[state][action] += self.alpha * td_error
        self.total_reward += reward
        self.rewards_history.append(reward)

print('Q-Learning Agent defined.')

### Step 8: Run Q-Learning Simulation

In [ ]:
np.random.seed(42)
ql_all_rewards = []

for uid in active_users[:50]:
    agent = QLearningAgent(n_states, n_arms, alpha=0.1, gamma=0.9, epsilon=0.1)
    rated = user_rated.get(uid, set())
    valid_mask = np.array([mid in rated for mid in all_movie_ids])
    if valid_mask.sum() == 0:
        continue

    rated_so_far = []
    state = get_user_state(uid, rated_so_far)

    for step in range(min(100, int(valid_mask.sum()))):
        action = agent.select_action(state, valid_mask)
        movie_id = all_movie_ids[action]
        reward = get_reward(uid, movie_id)

        # Update state after recommendation
        rated_so_far.append(movie_id)
        next_state = get_user_state(uid, rated_so_far)

        # Remove recommended movie
        valid_mask[action] = False
        agent.update(state, action, reward, next_state, valid_mask)
        state = next_state
        ql_all_rewards.append(reward)

ql_cum = np.cumsum(ql_all_rewards) / np.arange(1, len(ql_all_rewards) + 1)
print(f'Q-Learning: {len(ql_all_rewards)} steps, avg reward: {np.mean(ql_all_rewards):.4f}')

### Step 9: Compare All RL Approaches

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Cumulative average reward comparison
axes[0].plot(eg_cum, label='ε-Greedy (ε=0.1)', alpha=0.8)
axes[0].plot(ucb_cum, label='UCB (c=2)', alpha=0.8)
axes[0].plot(ql_cum, label='Q-Learning', alpha=0.8)
axes[0].set_xlabel('Step')
axes[0].set_ylabel('Cumulative Average Reward')
axes[0].set_title('RL Approaches: Cumulative Average Reward')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Hit rate (positive reward ratio)
methods = ['ε-Greedy', 'UCB', 'Q-Learning']
hit_rates = [
    np.mean([1 if r > 0 else 0 for r in eg_rewards]),
    np.mean([1 if r > 0 else 0 for r in ucb_rewards]),
    np.mean([1 if r > 0 else 0 for r in ql_all_rewards])
]
colors = ['#2196F3', '#FF9800', '#4CAF50']
axes[1].bar(methods, hit_rates, color=colors, edgecolor='black', alpha=0.8)
axes[1].set_ylabel('Hit Rate (rating ≥ 4)')
axes[1].set_title('Positive Recommendation Rate')
axes[1].set_ylim(0, 1)
axes[1].grid(True, alpha=0.3, axis='y')
for i, v in enumerate(hit_rates):
    axes[1].text(i, v + 0.02, f'{v:.3f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

print('\n' + '='*55)
print('RL APPROACHES COMPARISON')
print('='*55)
print(f'{"Method":<15s} | {"Avg Reward":>12s} | {"Hit Rate":>10s} | {"Steps":>8s}')
print('-'*55)
for name, rewards in [('ε-Greedy', eg_rewards), ('UCB', ucb_rewards), ('Q-Learning', ql_all_rewards)]:
    hr = np.mean([1 if r > 0 else 0 for r in rewards])
    print(f'{name:<15s} | {np.mean(rewards):>12.4f} | {hr:>10.4f} | {len(rewards):>8d}')

### Step 10: Compare RL with Traditional Models (SVD, User-CF, Item-CF)

Use simple baselines to compare against RL approaches.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import csr_matrix

# Build user-item matrix for traditional methods
user_ids = sorted(ratings['userId'].unique())
movie_ids = sorted(ratings['movieId'].unique())
uid_map = {u: i for i, u in enumerate(user_ids)}
mid_map = {m: i for i, m in enumerate(movie_ids)}

rows = ratings['userId'].map(uid_map).values
cols = ratings['movieId'].map(mid_map).values
vals = ratings['rating'].values
ui_matrix = csr_matrix((vals, (rows, cols)), shape=(len(user_ids), len(movie_ids)))
ui_dense = ui_matrix.toarray()

# Global mean for missing values
global_mean = ratings['rating'].mean()

# --- User-Based CF ---
user_sim = cosine_similarity(ui_dense)
np.fill_diagonal(user_sim, 0)

def predict_user_cf(uid_idx, mid_idx, k=20):
    sims = user_sim[uid_idx]
    top_k = np.argsort(sims)[-k:]
    rated_mask = ui_dense[top_k, mid_idx] > 0
    if rated_mask.sum() == 0:
        return global_mean
    w = sims[top_k][rated_mask]
    r = ui_dense[top_k, mid_idx][rated_mask]
    return np.average(r, weights=np.abs(w) + 1e-8)

# --- Item-Based CF ---
item_sim = cosine_similarity(ui_dense.T)
np.fill_diagonal(item_sim, 0)

def predict_item_cf(uid_idx, mid_idx, k=20):
    sims = item_sim[mid_idx]
    user_ratings = ui_dense[uid_idx]
    rated_items = np.where(user_ratings > 0)[0]
    if len(rated_items) == 0:
        return global_mean
    item_sims = sims[rated_items]
    top_k_idx = np.argsort(item_sims)[-k:]
    sel_items = rated_items[top_k_idx]
    w = sims[sel_items]
    r = user_ratings[sel_items]
    if np.abs(w).sum() == 0:
        return global_mean
    return np.average(r, weights=np.abs(w) + 1e-8)

# --- SVD ---
from numpy.linalg import svd as np_svd
ui_filled = ui_dense.copy()
user_means = np.true_divide(ui_dense.sum(1), (ui_dense > 0).sum(1) + 1e-8)
for i in range(ui_filled.shape[0]):
    ui_filled[i][ui_filled[i] == 0] = user_means[i]
U, sigma, Vt = np_svd(ui_filled, full_matrices=False)
k_svd = 50
svd_pred = U[:, :k_svd] @ np.diag(sigma[:k_svd]) @ Vt[:k_svd, :]
svd_pred = np.clip(svd_pred, 0.5, 5.0)

print('Traditional models computed.')

In [ ]:
# Evaluate traditional models on the same users/movies used in RL simulation
# For each RL recommendation, compute what traditional models would predict
np.random.seed(42)
trad_results = {'User-CF': [], 'Item-CF': [], 'SVD': []}
rl_actual_ratings = []

for uid in active_users[:50]:
    if uid not in uid_map:
        continue
    uid_idx = uid_map[uid]
    rated = user_rated.get(uid, set())
    # Sample same number of movies as RL did
    rated_list = list(rated)
    sample_size = min(100, len(rated_list))
    sampled = np.random.choice(rated_list, size=sample_size, replace=False)

    for mid in sampled:
        if mid not in mid_map:
            continue
        mid_idx = mid_map[mid]
        actual = reward_lookup.get((uid, mid), 0)
        rl_actual_ratings.append(actual)

        # Traditional predictions -> convert to reward scale
        for name, pred_fn in [('User-CF', lambda: predict_user_cf(uid_idx, mid_idx)),
                               ('Item-CF', lambda: predict_item_cf(uid_idx, mid_idx)),
                               ('SVD', lambda: svd_pred[uid_idx, mid_idx])]:
            pred_rating = pred_fn()
            pred_reward = 1.0 if pred_rating >= 4.0 else -1.0
            trad_results[name].append(pred_reward)

# Compute hit rates for traditional models
print('='*65)
print('COMPARISON: RL vs TRADITIONAL MODELS')
print('='*65)
print(f'{"Method":<15s} | {"Avg Reward":>12s} | {"Hit Rate":>10s} | {"Explore?":>10s}')
print('-'*65)

# RL methods
for name, rewards in [('ε-Greedy', eg_rewards), ('UCB', ucb_rewards), ('Q-Learning', ql_all_rewards)]:
    hr = np.mean([1 if r > 0 else 0 for r in rewards])
    print(f'{name:<15s} | {np.mean(rewards):>12.4f} | {hr:>10.4f} | {"Yes":>10s}')

print('-'*65)

# Traditional methods
for name in ['User-CF', 'Item-CF', 'SVD']:
    rewards = trad_results[name]
    hr = np.mean([1 if r > 0 else 0 for r in rewards])
    print(f'{name:<15s} | {np.mean(rewards):>12.4f} | {hr:>10.4f} | {"No":>10s}')

### Step 11: Exploration vs Exploitation Analysis

In [ ]:
# Measure exploration: how many unique movies each method recommends
np.random.seed(42)
unique_movies = {'ε-Greedy': set(), 'UCB': set(), 'Q-Learning': set()}

# Re-run to track unique movies recommended
for uid in active_users[:50]:
    rated = user_rated.get(uid, set())
    valid_mask = np.array([mid in rated for mid in all_movie_ids])
    if valid_mask.sum() == 0:
        continue
    n_steps = min(100, int(valid_mask.sum()))

    # ε-Greedy
    b = EpsilonGreedyBandit(n_arms, epsilon=0.1)
    vm = valid_mask.copy()
    for _ in range(n_steps):
        arm = b.select_arm(vm)
        unique_movies['ε-Greedy'].add(all_movie_ids[arm])
        b.update(arm, get_reward(uid, all_movie_ids[arm]))
        vm[arm] = False

    # UCB
    b = UCBBandit(n_arms, c=2.0)
    vm = valid_mask.copy()
    for _ in range(n_steps):
        arm = b.select_arm(vm)
        unique_movies['UCB'].add(all_movie_ids[arm])
        b.update(arm, get_reward(uid, all_movie_ids[arm]))
        vm[arm] = False

    # Q-Learning
    agent = QLearningAgent(n_states, n_arms, alpha=0.1, gamma=0.9, epsilon=0.1)
    vm = valid_mask.copy()
    rated_so_far = []
    state = get_user_state(uid, rated_so_far)
    for _ in range(n_steps):
        action = agent.select_action(state, vm)
        mid = all_movie_ids[action]
        reward = get_reward(uid, mid)
        rated_so_far.append(mid)
        next_state = get_user_state(uid, rated_so_far)
        vm[action] = False
        agent.update(state, action, reward, next_state, vm)
        state = next_state
        unique_movies['Q-Learning'].add(mid)

print('Exploration Analysis: Unique Movies Recommended')
print('='*50)
for name, s in unique_movies.items():
    print(f'{name:<15s}: {len(s):>5d} unique movies')

### Step 12: Epsilon Sensitivity Analysis

In [ ]:
# Test different epsilon values
epsilons = [0.0, 0.05, 0.1, 0.2, 0.3, 0.5]
eps_results = []

for eps in epsilons:
    np.random.seed(42)
    rewards, cum, _, _ = run_bandit_simulation(
        EpsilonGreedyBandit, {'epsilon': eps}, active_users[:50], n_steps=100)
    hr = np.mean([1 if r > 0 else 0 for r in rewards])
    eps_results.append((eps, np.mean(rewards), hr, cum))
    print(f'ε={eps:.2f}: avg reward={np.mean(rewards):.4f}, hit rate={hr:.4f}')

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for eps, avg_r, hr, cum in eps_results:
    axes[0].plot(cum, label=f'ε={eps}', alpha=0.7)
axes[0].set_xlabel('Step')
axes[0].set_ylabel('Cumulative Average Reward')
axes[0].set_title('ε-Greedy: Effect of Epsilon on Learning')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epsilons, [r[1] for r in eps_results], 'o-', label='Avg Reward', color='blue')
axes[1].set_xlabel('Epsilon (ε)')
axes[1].set_ylabel('Average Reward', color='blue')
ax2 = axes[1].twinx()
ax2.plot(epsilons, [r[2] for r in eps_results], 's-', label='Hit Rate', color='red')
ax2.set_ylabel('Hit Rate', color='red')
axes[1].set_title('ε-Greedy: Epsilon vs Performance')
axes[1].grid(True, alpha=0.3)
lines1, labels1 = axes[1].get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
axes[1].legend(lines1 + lines2, labels1 + labels2)

plt.tight_layout()
plt.show()

### Analysis

**Key Findings:**

1. **ε-Greedy (ε=0.1)**: Explores 10% of the time, exploits 90%. Simple and effective baseline. Performance is sensitive to ε — too high means too much random exploration, too low means the agent gets stuck on early estimates.

2. **UCB**: Systematically explores under-sampled movies using confidence bounds. Tends to explore more broadly early on, then converges to good recommendations. No ε parameter to tune.

3. **Q-Learning**: Learns state-dependent policies — different recommendations based on user genre preferences. Can adapt its strategy as it learns more about a user, but needs more data to converge due to the larger state-action space.

4. **RL vs Traditional**: Traditional models (CF, SVD) optimize for immediate rating prediction accuracy. RL models explicitly balance exploration (discovering new movies the user might like) with exploitation (recommending known good movies). This exploration capability is crucial for avoiding filter bubbles and improving long-term engagement.

5. **Exploration-Exploitation Tradeoff**: Higher exploration leads to more diverse recommendations but lower immediate reward. The optimal balance depends on the application — a new user benefits from more exploration, while a returning user prefers exploitation of known preferences.